<a href="https://colab.research.google.com/github/andrew-veriga/Titans_jax/blob/main/colabs/Titans_jax_TPUv6e-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# HuggingFace authentication
# !pip install -q huggingface_hub
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

# Gemma-Titans Training on TPU with Kauldron

This notebook demonstrates the complete pipeline for training and using the **Gemma-Titans** hybrid model on a Google Cloud TPU v5e using the `Kauldron` framework (DeepMind's standard training library).

### Steps included:
1. **Initialization:** Loading base Gemma weights and randomly initializing Titans NLTM modules using `SkipTitans`.
2. **Pre-training (CPT):** Training *only* the Titans memory layers using `kd.optim.partial_updates` to avoid TPU OOM, utilizing a proper dataset.
3. **Save/Load:** Splitting the PyTree to save only the fine-tuned memory weights.
4. **Dialogue Inference:** Running the model and updating the internal memory cache dynamically.

In [ ]:
# 0. Environment Setup

# Clone the new kauldron repository
!git clone --depth 1 https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
# Clone the gemma repository if not already present
!git clone --depth 1 https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone --depth 1 https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

!pip install -q flax optax seqio





In [ ]:
!git clone --depth 1 https://github.com/andrew-veriga/Titans_jax.git


In [ ]:
!pip install importlib_resources

In [ ]:
# !git clone --depth 1 https://github.com/jax-ml/jax.git
# !pip install -q ./jax

# Start

In [ ]:
import os
os._exit(0)

In [ ]:
# !git pull

In [ ]:
# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())
from kauldron import kd
from kauldron.ktyping import Array

In [ ]:
import jax
import jax.numpy as jnp
import optax
from kauldron import kd
import numpy as np
import os
import orbax.checkpoint as ocp
import dataclasses
import shutil

# Gemma imports
from gemma import gm
from gemma.gm.nn import _config

# Our custom Titans integration
import importlib

%cd Titans_jax
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils
from hf_checkpoint import (
    save_checkpoint_to_hf, load_checkpoint_from_hf,
    save_last_metadata, load_last_metadata,
    load_metadata_from_hf,
    reconstruct_opt_params, schedule,
)

# ═══════════════════════════════════════════════════════════
# FIRST_RUN = True  → random init (first ever training)
# FIRST_RUN = False → load Phase 1 checkpoint from HF
# ═══════════════════════════════════════════════════════════
FIRST_RUN = True

HF_CKPT_REPO = "veriga/titans-checkpoints"

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")
print(f"FIRST_RUN = {FIRST_RUN}")
""" старые настройки
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"
"""
# Разрешаем JAX забрать память сразу для максимальной скорости
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"

# Увеличиваем долю памяти (оставляем чуть-чуть на системные нужды)
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".95"

# Оптимизируем флаги для производительности, а не для экономии
os.environ["XLA_FLAGS"] = (
    "--xla_tpu_enable_data_parallel_all_reduce_opt=true "
    "--xla_tpu_joint_all_gather_opt=true "
    "--xla_tpu_enable_latency_hiding_scheduler=true " # Скрывает задержки памяти за вычислениями
    "--xla_tpu_all_reduce_combine_threshold_bytes=134217728" # Оптимально для больших батчей
)

os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"


## 1. Model initialization

We load previously trained titans weights and merge them with original Gemma weights

### hyper-parameters

In [ ]:
batch_size = 16
max_length = 1024 #
total_steps = 40000

titans_phase2_first_layer = 17  # Titans layers from this index onward are active.
                                 # Earlier layers revert to standard Gemma blocks.
                                 # 17 → layers 17,23 active (~25GB compile RAM)
                                 # 23 → layer 23 only  (~5GB compile RAM)

# Гиперпараметры Titans Memory
huber_loss_delta = optax.constant_schedule(0.1)

config = Gemma_Titans_Config.from_gemma_config()
_all_titans_layers = config.titans_layer_indices

print(f"All Titans layers: {_all_titans_layers}")
active_titans_layers = tuple(l for l in _all_titans_layers if l >= titans_phase2_first_layer)
print(f"Active Titans layers: {active_titans_layers}")


In [ ]:
# Define the model configuration
import dataclasses

experimental_config = {
            'heads': 8,
            'dim_head': 128,
            'chunk_size': 32,
            'mlp_depth': 2,
            'max_grad_norm': 0.5,
            'elastic_net_lambda': 0.005,
            'diff_view': False,
            'is_look_ahead': False,
            'huber_loss_delta': 0.5,
        }

WARMUP = 1500

b1_schedule = optax.linear_schedule(
    init_value=0.7,
    end_value=0.90,
    transition_steps=2000,
    transition_begin=WARMUP
)

opt_params = {
    "lr_muon": optax.warmup_cosine_decay_schedule(
        init_value=5e-4,
        peak_value=5e-4,
        warmup_steps=WARMUP,
        decay_steps=total_steps - WARMUP,
        end_value=1e-5
	),
	"beta": 0.90,
	"lr_adam": optax.warmup_cosine_decay_schedule(
        init_value=1e-4,
        peak_value=1e-4,
        warmup_steps=WARMUP,
        decay_steps=total_steps - WARMUP,
        end_value=1e-5,
	),
	"adam_b1": b1_schedule,
    "adam_b2": 0.85,
    "lr_gate": optax.warmup_cosine_decay_schedule(
	    init_value=5e-3,
	    peak_value=5e-3,
	    warmup_steps=WARMUP,
	    decay_steps=total_steps - WARMUP,
	    end_value=5e-4,
	),
    "gate_b1": b1_schedule,
    "gate_b2": 0.95,
    "every_k_schedule": 8
}


In [ ]:
def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()
    titans_params = checkpointer.restore(load_path)
    return titans_params

merged_params = None
titans_delta_path = "./saved_titans_delta"
workdir = os.path.abspath("./titans_workdir_squad")
workdir_checkpoints = os.path.join(workdir, "checkpoints")

if os.path.exists(workdir_checkpoints) and len(os.listdir(workdir_checkpoints)) > 0:
    print(f"📁 Найдена директория {workdir_checkpoints}. Пропускаем загрузку весов.")
    print("Kauldron автоматически загрузит последнее состояние при старте обучения.")
elif not FIRST_RUN:
    # Повторный запуск: загружаем чекпойнт Phase 1 из HuggingFace
    last_meta = load_last_metadata(
        repo_id=HF_CKPT_REPO,
        phase=2,
        token=userdata.get('HF_TOKEN'),
    )

    if last_meta is not None:
        # Восстанавливаем experimental_config (можно использовать как neural_mem_kwargs)
        experimental_config = last_meta.get("experimental_config", experimental_config)
        print(f"📋 Restored experimental_config: {experimental_config}")

        # Восстанавливаем opt_params: schedules → callable
        if "opt_params" in last_meta:
            opt_params = reconstruct_opt_params(last_meta["opt_params"])
            print(f"📋 Restored opt_params with schedules: {list(opt_params.keys())}")

        # Авто-определение параметров чекпойнта
        titans_phase2_first_layer = last_meta.get("first_layer", titans_phase2_first_layer)
        total_steps = last_meta.get("total_steps", 10)
        print(f"📋 Checkpoint: {last_meta.get('checkpoint')}")
    else:
        print("⚠️ Last metadata not found — using current notebook values")


    ckpt_dir = load_checkpoint_from_hf(
        repo_id=HF_CKPT_REPO,
        phase=1,
        first_layer=titans_phase2_first_layer,
        total_steps=total_steps,
        local_dir=".",
    )


    if ckpt_dir is not None:
        print("Loading original Gemma weights...")
        original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

        print("Merging Titans weights from HF checkpoint...")
        loaded_titans_params = load_titans_weights(ckpt_dir)
        merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params, remove_dead_attn=False)
        print("✅ Success: Phase 1 Titans weights loaded from HF and merged.")
    else:
        print("⚠️ Чекпойнт Phase 1 не найден на HF. Titans layers будут инициализированы случайно.")
else:
    print("🆕 FIRST_RUN = True. Titans layers будут инициализированы случайно через SkipTitans.")

## 2. Pre-training (CPT)

We use `Kauldron` to orchestrate the training.
- **Dataset:** Pre-tokenized OpenWebText from HuggingFace.
- **Model Config:** We use the standard `gm.nn.Gemma3_1B.config`.
- **SkipTitans:** Handles loading Gemma weights while keeping Titans randomized.
- **partial_updates:** Ensures XLA only builds backprop graphs for memory parameters to prevent OOM.

In [ ]:
tokenizer = gm.text.Gemma3Tokenizer()

titans_config = dataclasses.replace(
    Gemma3_1B_Titans.config,
    training_phase=1,
    titans_layer_indices=active_titans_layers,
    titans_phase2_first_layer=titans_phase2_first_layer,
    neural_mem_kwargs = experimental_config
)
model = Gemma3_1B_Titans(
    config=titans_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
)

In [ ]:
# !git pull

In [ ]:
# import gemma_titans
# importlib.reload(gemma_titans)
# from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config

In [ ]:
# import importlib
# from adam_atan2 import adam_atan2

## dataset


### Pre-tokenized OpenWebText from HuggingFace

In [ ]:
from fast_dataset_loader import get_fast_openwebtext

train_ds = get_fast_openwebtext(
    repo_id="veriga/openwebtext-gemma3-tokenized-1024",
    split="train",
    batch_size=batch_size,
    shuffle=True,
    max_length=max_length,
    num_epochs=None,
    cache_dir = "/content/drive/Shareddrives/shared_veriga/jax_cache"
)
print("✅ Dataset ready (pre-tokenized, no tokenizer needed).")

### optimizer routing

In [ ]:
from routing_optimizer import make_routing_optimizer

routing_optimizer = make_routing_optimizer(opt_params)

In [ ]:
step = 500
print(f"lr_gate {opt_params['lr_gate'](step)}")
print(f"lr_muon {opt_params['lr_muon'](step)}")
print(f"lr_adam {opt_params['lr_adam'](step)}")
print(f"huber {huber_loss_delta(step)}")

### metrics

In [ ]:
import flax
import flax.linen as nn
from kauldron import metrics as kd_metrics
from kauldron import kontext

class FullParamsInit(kd.ckpts.InitTransform):
    def __init__(self, params):
        self.params = params
    def transform(self, state: kd.train.TrainState) -> kd.train.TrainState:
        return state.replace(params=self.params)

# LM loss
# for layer in active_titans_layers:
#     train_losses[f"mem_loss/layer_{layer}"] = kd.losses.Value(
#         values=f"preds.layer_losses['mem_loss_layer_{layer}']"
#     )
# preds.layer_losses['mem_loss_layer_17']
# Did you mean one of:
#   'preds.layer_losses.lm_loss'
#   'preds.layer_losses.loss_layer_17'
#   'preds.layer_losses.loss_layer_23'
train_losses = {
    f"lm_loss_{layer}": kd.losses.Value(
        values=f"preds.layer_losses['loss_layer_{layer}']"
    ) for layer in active_titans_layers
}

# Метрики
@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class MemoryGateMetric(kd_metrics.Metric):
    params: kd.kontext.Key = "params"

    @flax.struct.dataclass
    class State(kd_metrics.State):
        gate_metrics: flax.core.FrozenDict[str, jnp.ndarray] = flax.core.FrozenDict()
        def compute(self):
            return {k: np.array(v, dtype=np.float32) for k, v in self.gate_metrics.items()}
        @classmethod
        def empty(cls):
            return cls(gate_metrics=flax.core.FrozenDict())
        def merge(self, other):
            return self

    def get_state(self, params=None, **kwargs) -> State:
        if params is None:
            return self.State.empty()
        stats_dict = {}
        def find_gates(tree, stats_dict: dict, path_prefix: str = ""):
            for key, val in tree.items():
                current_path = f"{path_prefix}_{key}" if path_prefix else str(key)
                if key == "memory_gate_proj":
                    leaves = jax.tree_util.tree_leaves(val)
                    all_params = jnp.concatenate([jnp.ravel(p) for p in leaves])
                    mean_val = jnp.mean(all_params)
                    openness = jax.nn.sigmoid(mean_val)
                    stats_dict[f"titans_gates/{current_path}_raw_mean"] = mean_val
                    stats_dict[f"titans_gates/{current_path}_openness"] = openness
                    std_val = jnp.std(all_params)
                    stats_dict[f"titans_gates/{current_path}_raw_std"] = std_val
                    if isinstance(val, (dict, flax.core.FrozenDict)):
                        actual_weights = val.get('kernel', val)
                    else:
                        actual_weights = val
                    mean_val = jnp.mean(actual_weights)
                    openness = jax.nn.sigmoid(mean_val)
                    stats_dict[f"titans_gates/{current_path}_raw_mean"] = mean_val
                    stats_dict[f"titans_gates/{current_path}_openness"] = openness
                    stats_dict[f"titans_gates/{current_path}_raw_std"] = jnp.std(actual_weights)
                elif isinstance(val, (dict, flax.core.FrozenDict)):
                    find_gates(val, stats_dict, current_path)
        return self.State(gate_metrics=flax.core.freeze(stats_dict))

@flax.struct.dataclass
class LRState(kd_metrics.State):
    lr_value: jnp.ndarray
    @classmethod
    def empty(cls):
        return cls(lr_value=jnp.array(0.0))
    def merge(self, other):
        return self
    def compute(self):
        return self.lr_value

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class HuberDeltaMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(huber_loss_delta(step)))
@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class GateLearningRateMetric(kd_metrics.Metric):
    step: kontext.Key = "step"
    def get_state(self, step, **kwargs):
        return LRState(lr_value=jnp.array(opt_params['lr_gate'](step)))

class TPUMemoryMetric(kd_metrics.Metric):
    """Метрика для логирования использования памяти TPU в ГБ."""
    @flax.struct.dataclass
    class State(kd_metrics.State):
        def compute(self):
            stats_dict = {}
            for i, device in enumerate(jax.devices()):
                try:
                    m_stats = device.memory_stats()
                    if not m_stats:
                        continue
                    prefix = f"device_{i}"
                    if 'bytes_in_use' in m_stats:
                        used_gb = m_stats['bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/used_gb"] = np.array(used_gb, dtype=np.float32)
                    if 'peak_bytes_in_use' in m_stats:
                        peak_gb = m_stats['peak_bytes_in_use'] / 1e9
                        stats_dict[f"{prefix}/peak_gb"] = np.array(peak_gb, dtype=np.float32)
                    if 'limit' in m_stats and 'bytes_in_use' in m_stats:
                        limit_gb = m_stats['limit'] / 1e9
                        usage_pct = (m_stats['bytes_in_use'] / m_stats['limit']) * 100
                        stats_dict[f"{prefix}/usage_pct"] = np.array(usage_pct, dtype=np.float32)
                except (AttributeError, ValueError, RuntimeError):
                    pass
            return stats_dict
        @classmethod
        def empty(cls):
            return cls()
        def merge(self, other):
            return self

    def get_state(self, **kwargs) -> State:
        return self.State().empty()

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class Perplexity(kd.metrics.Metric):
    loss: kd.kontext.Key = "preds.layer_losses['lm_loss']"
    @flax.struct.dataclass
    class State(kd.metrics.base_state.AverageState):
        def compute(self):
            mean_loss = super().compute()
            return jnp.exp(mean_loss)
    def get_state(self,*, loss) -> State:
        return self.State.from_values(values=loss)

@dataclasses.dataclass(kw_only=True, frozen=True, eq=True)
class LMAccuracy(kd.metrics.Metric):
    acc: kd.kontext.Key = "preds.layer_losses['lm_accuracy']"
    @flax.struct.dataclass
    class State(kd.metrics.base_state.AverageState):
        pass
    def get_state(self,*, acc) -> State:
        return self.State.from_values(values=acc)

train_metrics = {
    "LM/lr_gate": GateLearningRateMetric(),
    "tpu_memory": TPUMemoryMetric()
}
train_summaries = {}
for layer in active_titans_layers:
    key = f"Gates_Dist_{layer}"
    train_summaries[key] = kd.summaries.HistogramSummary(
        tensor=f"params.layer_{layer}.memory_gate_proj.kernel"
    )

In [ ]:
import flax.linen as nn
import grain

trainer = kd.train.Trainer(
    seed=42,
    workdir=workdir,
    train_ds=train_ds,
    model=model,
    init_transform=(
        FullParamsInit(merged_params)
        if ('merged_params' in dir()) and (merged_params is not None)
        else SkipTitans(
            wrapped=gm.ckpts.LoadCheckpoint(
                path=gm.ckpts.CheckpointPath.GEMMA3_1B_IT,
            ),
            workdir=os.path.abspath('./SkipTitans_workdir'),
        )
    ),
    checkpointer=kd.ckpts.Checkpointer(
        save_interval_steps=500,
    ),
    num_train_steps=total_steps,
    train_losses=train_losses,
    train_metrics=train_metrics,
    train_summaries=train_summaries,
    optimizer=routing_optimizer,
)

print("Trainer initialized. Starting compilation and training...")


### 2.5 Monitoring with TensorBoard

In [ ]:
%reload_ext tensorboard


In [ ]:
from tensorboard import notebook
notebook.list()

In [ ]:
!rm -rf /tmp/.tensorboard-info/*
!fuser -k 6006/tcp

In [ ]:
%tensorboard --logdir ./titans_workdir_squad/ --port=6006

## Start training

In [ ]:
state, aux = trainer.train()

## 3. Save Trained Weights to HuggingFace
We don't want to save the entire 1B model, just our new memory projections.

In [ ]:
def save_titans_weights(state: kd.train.TrainState, save_dir: str):
    _, titans_params = titans_tree_utils.split_titans_params(state.params)
    save_path = os.path.abspath(save_dir)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(save_path, titans_params)
    checkpointer.wait_until_finished()
    print(f"Saved ONLY Titans weights to {save_path}")

new_weights_name = f"saved_titans_phase1_from_{titans_phase2_first_layer}"
save_titans_weights(state, f"./{new_weights_name}")

from google.colab import userdata

# 1. Загружаем чекпойнт (веса + метадата)
save_checkpoint_to_hf(
    save_dir=f"./{new_weights_name}",
    repo_id=HF_CKPT_REPO,
    phase=2,  # или 1 для TPUv6e-1
    experimental_config=experimental_config,
    opt_params=opt_params,
    first_layer=titans_phase2_first_layer,
    total_steps=total_steps,
    token=userdata.get('HF_TOKEN'),
)

# 2. Обновляем «указатель» на последний эксперимент
save_last_metadata(
    repo_id=HF_CKPT_REPO,
    phase=2,  # или 1
    first_layer=titans_phase2_first_layer,
    total_steps=total_steps,
    experimental_config=experimental_config,
    opt_params=opt_params,
    token=userdata.get('HF_TOKEN'),
)
print(f"✅ Phase 1 checkpoint + metadata uploaded to {HF_CKPT_REPO}")

In [ ]:
!ls ./{new_weights_name}

In [ ]:
# Upload checkpoint to HuggingFace Hub
from google.colab import userdata

save_checkpoint_to_hf(
    save_dir=f"./{new_weights_name}",
    repo_id=HF_CKPT_REPO,
    phase=1,
    metadata={
        "experimental_config": experimental_config,
        "opt_params": opt_params,
    },
    first_layer=titans_phase2_first_layer,
    total_steps=total_steps,
    token=userdata.get('HF_TOKEN'),
)
print(f"✅ Phase 1 checkpoint uploaded to {HF_CKPT_REPO}")

## Очистка TPU для повторного запуска (если нужно)

In [ ]:
import jax

def print_tpu_mem_tpu_native():
    d = jax.devices()[0]
    ms = d.memory_stats() or {}

    used = ms.get("bytes_in_use", 0)
    peak = ms.get("peak_bytes_in_use", 0)

    limit = ms.get("bytes_limit", 0)
    reserved = ms.get("bytes_reserved", 0)
    reservable_limit = ms.get("bytes_reservable_limit", 0)
    largest_free = ms.get("largest_free_block_bytes", 0)

    print(f"Device: {d}")
    print(f"used_gb:               {used / 1e9:.2f}")
    print(f"peak_used_gb:          {peak / 1e9:.2f}")
    print(f"bytes_limit_gb:        {limit / 1e9:.2f}")
    print(f"bytes_reserved_gb:     {reserved / 1e9:.2f}")
    print(f"reservable_limit_gb:   {reservable_limit / 1e9:.2f}")
    print(f"largest_free_block_gb: {largest_free / 1e9:.2f}")

    if reservable_limit > 0:
        reservable_free = max(reservable_limit - reserved, 0)
        print(f"reservable_free_gb:    {reservable_free / 1e9:.2f}")

print_tpu_mem_tpu_native()

In [ ]:
import jax
import gc

# 1. DESTROY the Python references to the old TPU buffers FIRST
try:
    del state
    del aux
except NameError:
    pass

# 2. Clear JAX's internal live buffer cache
for device in jax.devices():
    if hasattr(device, 'live_arrays'): device.live_arrays().clear()
    if hasattr(device, 'live_buffers'): device.live_buffers().clear()
    if hasattr(device, 'default_memory_tracker'): device.default_memory_tracker().clear()

jax.clear_caches()

# 3. NOW run Python garbage collection
gc.collect()

print("TPU memory cache cleared and garbage collection finished.")

print_tpu_mem_tpu_native()

## Повторный запуск

In [ ]:
trainer = trainer.replace(num_train_steps=50000)

state, aux = trainer.train()